In [ ]:
from pypokerengine.engine.card import Card
from pypokerengine.utils.card_utils import estimate_hole_card_win_rate

# # Function to generate cards
# def gen_cards(cards_str):
#     """
#     Generate Card objects from a list of card strings.
#     Card strings must be in the format 'CA' (e.g., 'CA' = Ace of Clubs).
#     """
#     suit_map = {
#         "C": Card.CLUB,
#         "H": Card.HEART,
#         "S": Card.SPADE,
#         "D": Card.DIAMOND
#     }
#     rank_map = {
#         "2": 2, "3": 3, "4": 4, "5": 5, "6": 6, "7": 7, "8": 8, "9": 9, 
#         "T": 10, "J": 11, "Q": 12, "K": 13, "A": 14
#     }

#     cards = []
#     for card_str in cards_str:
#         suit = suit_map[card_str[1].upper()]  # Second character is the suit
#         rank = rank_map[card_str[0].upper()]  # First character is the rank
#         cards.append(Card(suit, rank))

#     return cards
def gen_cards(cards_str):
    """
    Generate Card objects from a list of card strings.
    Card strings must be in the format 'CA' (e.g., 'CA' = Ace of Clubs).
    """
    suit_map = {
        "C": Card.CLUB,
        "H": Card.HEART,
        "S": Card.SPADE,
        "D": Card.DIAMOND
    }
    rank_map = {
        "2": 2, "3": 3, "4": 4, "5": 5, "6": 6, "7": 7, "8": 8, "9": 9, 
        "10": 10, "T": 10, "J": 11, "Q": 12, "K": 13, "A": 14
    }

    try:
        cards = []
        for card_str in cards_str:
            if len(card_str) == 3:  # Handle "10" case, e.g., "10S"
                rank = "10"
                suit = card_str[2]
            else:
                rank = card_str[0]  # Extract the rank (first character)
                suit = card_str[1]  # Extract the suit (second character)
            
            suit = suit_map[suit.upper()]  # Translate suit to Card enum
            rank = rank_map[rank.upper()]  # Translate rank to integer
            
            cards.append(Card(suit, rank))

        return cards
    except Exception as e:
        print(f"Error in gen_cards: {e}")
        return []

# Example Usage
if __name__ == "__main__":
    # Define your hole cards and community cards
    hole_cards = gen_cards(["JD", "3D"])  # Ace of Spades, King of Spades
    community_cards = gen_cards([])  # Queen of Spades, Jack of Spades, Ten of Spades

    # Estimate equity
    equities = {'2': None, '3':None, '4':None, '5':None, '6':None}
    for i in range(2,7):
        equity = estimate_hole_card_win_rate(
            nb_simulation=1000,
            nb_player=i,
            hole_card=hole_cards,
            community_card=community_cards
        )
        equities[str(i)]=(round(equity * 100, 2))  # Append rounded equity values to the list
    print(equities)
    # print(f"Estimated equity: {equity * 100:.2f}%")
  

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException
import re
import json
import logging
import torch
from poker_env import (
    GameState,
    SimpleHoldemEnv,
    ACTION_FOLD,
    ACTION_CHECK,
    ACTION_CALL,
    ACTION_RAISE_SMALL,
    ACTION_RAISE_MEDIUM,
    ACTION_ALL_IN,
    ACTION_SEQ_LEN,
    NUM_ACTIONS,
    STREET_PREFLOP,
    STREET_FLOP,
    STREET_TURN,
    STREET_RIVER,
)
from abstraction import encode_state
from networks import PolicyNet
from config import SMALL_BLIND, BIG_BLIND, STACK_SIZE
import winsound
import time
import random  # Add this import at the top of your script
from pypokerengine.engine.card import Card
from pypokerengine.utils.card_utils import estimate_hole_card_win_rate
from pokerutil import initialize_display, update_interactive_display

initialize_display()

POLICY_PATH = r"C:\\Users\\PRABAL YADAV\\Desktop\\machine learning iim\\pokerbotPlayOnline\\models\\policy phase3_120.pt"
TABLE_NUM_PLAYERS = 6
TABLE_SMALL_BLIND = SMALL_BLIND
TABLE_BIG_BLIND = BIG_BLIND
USE_POLICY_ONLY = True
FORCE_HERO_SEAT_NUMBER = 4
LOG_FULL_STATE = True
LOG_TABLE_TO_FILE = True
LOG_PRINT_CONSOLE = True
LOG_FILE_PATH = "table_state_debug.log"
LOG_EVERY_N = 1

logger = logging.getLogger("table_debug")
LOG_TICK = 0

ACTION_ID_TO_NAME = {
    ACTION_FOLD: "FOLD",
    ACTION_CHECK: "CHECK",
    ACTION_CALL: "CALL",
    ACTION_RAISE_SMALL: "RAISE_SMALL",
    ACTION_RAISE_MEDIUM: "RAISE_MEDIUM",
    ACTION_ALL_IN: "ALL_IN",
}

SEAT_SELECTORS = [
    ".table-6-players .player-area",
    ".table .player-area",
    ".player-area",
]
STACK_SELECTORS = [
    ".player-nameplate .text-block.amount",
    ".player-nameplate .amount",
]
BET_SELECTORS = [
    ".player-bet .amount-cont .amount",
    ".player-bet .amount",
]
NAME_SELECTORS = [
    ".player-nameplate .target",
    ".player-nameplate .nickname .target",
    ".player-nameplate .nickname",
]
BUTTON_SELECTORS = [
    ".game-position:not(.pt-visibility-hidden) .dealer.table-assets-btn-dealer",
    ".dealer.table-assets-btn-dealer",
]
SB_SELECTORS = [".small-blind", ".sb"]
BB_SELECTORS = [".big-blind", ".bb"]
ACTIVE_SELECTORS = [
    ".turn-to-act-indicator",
    ".timeout-wrapper",
    ".nameplate-blink",
    ".text-countdown",
]


def parse_money(text):
    if not text:
        return 0.0
    cleaned = text.replace(",", "").strip()
    match = re.search(r"-?\d+(?:\.\d+)?", cleaned)
    if not match:
        return 0.0
    try:
        return float(match.group(0))
    except ValueError:
        return 0.0


def extract_text(elem, selectors):
    for sel in selectors:
        try:
            txt = elem.find_element(By.CSS_SELECTOR, sel).text.strip()
            if txt:
                return txt
        except Exception:
            continue
    return ""


def is_hidden_by_class(elem):
    class_name = (elem.get_attribute("class") or "").lower()
    return "pt-hidden" in class_name or "pt-visibility-hidden" in class_name


def has_visible_child(elem, selectors):
    for sel in selectors:
        try:
            children = elem.find_elements(By.CSS_SELECTOR, sel)
        except Exception:
            children = []
        for child in children:
            try:
                if child.is_displayed():
                    return True
            except Exception:
                pass
            if not is_hidden_by_class(child):
                return True
    return False

def extract_seat_index(elem, fallback_idx):
    class_name = elem.get_attribute("class") or ""
    match = re.search(r"player-seat-(\d+)", class_name)
    if match:
        seat_num = int(match.group(1))
        if seat_num > 0 and seat_num <= TABLE_NUM_PLAYERS:
            return seat_num - 1
    for attr in ["data-seat", "data-seat-id", "data-position", "data-index", "data-seatindex"]:
        val = elem.get_attribute(attr)
        if val and val.isdigit():
            seat_num = int(val)
            if seat_num > 0 and seat_num <= TABLE_NUM_PLAYERS:
                return seat_num - 1
            return seat_num
        if val:
            match = re.search(r"(\d+)", val)
            if match:
                seat_num = int(match.group(1))
                if seat_num > 0 and seat_num <= TABLE_NUM_PLAYERS:
                    return seat_num - 1
                return seat_num
    id_attr = elem.get_attribute("id") or ""
    match = re.search(r"(\d+)", id_attr)
    if match:
        seat_num = int(match.group(1))
        if seat_num > 0 and seat_num <= TABLE_NUM_PLAYERS:
            return seat_num - 1
        return seat_num
    return fallback_idx

def read_seat_elements(driver):
    best = []
    for sel in SEAT_SELECTORS:
        try:
            elems = driver.find_elements(By.CSS_SELECTOR, sel)
        except Exception:
            elems = []
        if len(elems) > len(best):
            best = elems
    return best

def normalize_seats(seats, num_players):
    seat_map = {s.get("seat_index"): s for s in seats if s.get("seat_index") is not None}
    normalized = []
    for idx in range(num_players):
        if idx in seat_map:
            s = seat_map[idx]
        else:
            s = {
                "seat_index": idx,
                "name": "",
                "stack": 0.0,
                "bet": 0.0,
                "folded": True,
                "is_hero": False,
                "is_button": False,
                "is_sb": False,
                "is_bb": False,
                "is_active": False,
            }
        normalized.append(s)
    return normalized

def parse_seat_elem(elem, fallback_idx):
    seat_index = extract_seat_index(elem, fallback_idx)
    name_text = extract_text(elem, NAME_SELECTORS)
    stack_text = extract_text(elem, STACK_SELECTORS)
    bet_text = extract_text(elem, BET_SELECTORS)

    class_name = (elem.get_attribute("class") or "").lower()
    is_hero = "my-player" in class_name

    action_text = extract_text(elem, [".player-action .action-text"])
    action_text_lower = action_text.lower()

    is_folded_action = "fold" in action_text_lower
    is_sit_out_action = "sit out" in action_text_lower or "sit-out" in action_text_lower
    has_fold_class = has_visible_child(elem, [".player-action.action-fold"])
    is_inactive = "inactive" in class_name or "sit-out" in class_name or "player-sit-out" in class_name

    is_folded = is_folded_action or is_sit_out_action or has_fold_class or is_inactive

    is_button = has_visible_child(elem, BUTTON_SELECTORS)
    is_sb = has_visible_child(elem, SB_SELECTORS)
    is_bb = has_visible_child(elem, BB_SELECTORS)
    is_active = has_visible_child(elem, ACTIVE_SELECTORS)

    return {
        "seat_index": seat_index,
        "name": name_text,
        "raw_name_text": name_text,
        "raw_stack_text": stack_text,
        "raw_bet_text": bet_text,
        "class_name": class_name,
        "action_text": action_text,
        "stack": parse_money(stack_text),
        "bet": parse_money(bet_text),
        "folded": bool(is_folded),
        "is_hero": bool(is_hero),
        "is_button": bool(is_button),
        "is_sb": bool(is_sb),
        "is_bb": bool(is_bb),
        "is_active": bool(is_active),
    }

def resolve_positions(seats, hero_index):
    n = len(seats)
    button_idx = next((i for i, s in enumerate(seats) if s["is_button"]), None)
    sb_idx = next((i for i, s in enumerate(seats) if s["is_sb"]), None)
    bb_idx = next((i for i, s in enumerate(seats) if s["is_bb"]), None)

    if button_idx is None:
        button_idx = hero_index if hero_index is not None else 0
    if sb_idx is None:
        sb_idx = (button_idx + 1) % n if n else 0
    if bb_idx is None:
        bb_idx = (sb_idx + 1) % n if n else 0

    return button_idx, sb_idx, bb_idx


def resolve_to_act(seats, hero_index, force_hero_to_act=False):
    active_idx = next((i for i, s in enumerate(seats) if s["is_active"]), None)
    if active_idx is not None:
        return active_idx
    if force_hero_to_act:
        return hero_index
    return None


def read_table_snapshot(driver, force_hero_to_act=False):
    seat_elems = read_seat_elements(driver)
    if not seat_elems:
        print("No seat elements found; update SEAT_SELECTORS.")
        log_event("No seat elements found", {"selectors": SEAT_SELECTORS})
        return None
    seats_raw = [parse_seat_elem(elem, idx) for idx, elem in enumerate(seat_elems)]
    num_players = TABLE_NUM_PLAYERS if TABLE_NUM_PLAYERS else max([s.get("seat_index", 0) for s in seats_raw] + [0]) + 1
    seats = normalize_seats(seats_raw, num_players)
    hero_index = next((i for i, s in enumerate(seats) if s["is_hero"]), None)
    if FORCE_HERO_SEAT_NUMBER:
        forced = max(0, min(num_players - 1, FORCE_HERO_SEAT_NUMBER - 1))
        hero_index = forced
        seats[hero_index]["is_hero"] = True
    if hero_index is None:
        hero_index = 0
    button_idx, sb_idx, bb_idx = resolve_positions(seats, hero_index)
    to_act = resolve_to_act(seats, hero_index, force_hero_to_act=force_hero_to_act)
    return {
        "seats": seats,
        "hero_index": hero_index,
        "button": button_idx,
        "sb": sb_idx,
        "bb": bb_idx,
        "to_act": to_act,
        "num_players": num_players,
        "seat_elem_count": len(seat_elems),
    }

def street_from_board(community_cards):
    count = len(community_cards)
    if count >= 5:
        return STREET_RIVER
    if count == 4:
        return STREET_TURN
    if count == 3:
        return STREET_FLOP
    return STREET_PREFLOP


def card_str_to_id(card_str):
    if not card_str:
        return None
    text = card_str.strip().upper()
    if len(text) == 3:
        rank = text[:2]
        suit = text[2:]
    else:
        rank = text[0]
        suit = text[1]

    rank_map = {
        "2": 2, "3": 3, "4": 4, "5": 5, "6": 6, "7": 7, "8": 8, "9": 9,
        "10": 10, "T": 10, "J": 11, "Q": 12, "K": 13, "A": 14,
    }
    suit_map = {"S": 0, "H": 1, "D": 2, "C": 3}
    if rank not in rank_map or suit not in suit_map:
        return None
    return suit_map[suit] * 13 + (rank_map[rank] - 2)


def cards_str_to_ids(cards):
    ids = []
    for c in cards:
        card_id = card_str_to_id(c)
        if card_id is not None:
            ids.append(card_id)
    return ids


class TableTracker:
    def __init__(self):
        self.prev_contrib = []
        self.prev_folded = []
        self.prev_pot = 0.0
        self.prev_street = None
        self.action_seq = []
        self.initial_stacks = []
        self.last_aggressor = -1
        self.players_acted = []

    def reset_hand(self, seats):
        self.action_seq = []
        self.initial_stacks = [s["stack"] + s["bet"] for s in seats]
        self.last_aggressor = -1

    def _push_action(self, player, action, invested, pot_before):
        denom = max(1.0, pot_before)
        size_norm = 0.0
        if invested > 0.0:
            size_norm = min(invested / denom, 4.0) / 4.0
        self.action_seq.append((player, action, size_norm))
        if len(self.action_seq) > ACTION_SEQ_LEN:
            self.action_seq = self.action_seq[-ACTION_SEQ_LEN:]
        if action in (ACTION_RAISE_SMALL, ACTION_RAISE_MEDIUM, ACTION_ALL_IN):
            self.last_aggressor = player

    def update(self, seats, pot, street, to_act):
        n = len(seats)
        if n <= 0:
            return
        if self.prev_street is None or street < self.prev_street or (street == STREET_PREFLOP and pot < self.prev_pot):
            self.reset_hand(seats)

        if len(self.prev_contrib) != n:
            self.prev_contrib = [s["bet"] for s in seats]
            self.prev_folded = [s["folded"] for s in seats]
        else:
            prev_bet = max(self.prev_contrib) if self.prev_contrib else 0.0
            curr_contrib = [s["bet"] for s in seats]
            curr_bet = max(curr_contrib) if curr_contrib else 0.0

            for i in range(n):
                if not self.prev_folded[i] and seats[i]["folded"]:
                    self._push_action(i, ACTION_FOLD, 0.0, self.prev_pot)
                if curr_contrib[i] > self.prev_contrib[i]:
                    invested = curr_contrib[i] - self.prev_contrib[i]
                    prev_stack = seats[i]["stack"] + invested
                    if seats[i]["stack"] <= 1e-9:
                        action = ACTION_ALL_IN
                    elif curr_contrib[i] >= curr_bet and curr_bet > prev_bet and curr_contrib[i] == curr_bet:
                        if invested <= 0.5 * max(1.0, pot):
                            action = ACTION_RAISE_SMALL
                        else:
                            action = ACTION_RAISE_MEDIUM
                    else:
                        action = ACTION_CALL
                    self._push_action(i, action, invested, self.prev_pot)

            self.prev_contrib = curr_contrib
            self.prev_folded = [s["folded"] for s in seats]

        self.prev_pot = pot
        self.prev_street = street
        self.players_acted = [True] * n
        if to_act is not None and 0 <= to_act < n:
            self.players_acted[to_act] = False


def build_game_state(snapshot, hole_cards, community_cards, pot, tracker):
    seats = snapshot.get("seats", [])
    if not seats:
        return None, None
    hero_index = snapshot.get("hero_index", 0)
    to_act = snapshot.get("to_act")
    if to_act is None:
        to_act = hero_index

    street = street_from_board(community_cards)
    tracker.update(seats, pot, street, to_act)

    stacks = [s["stack"] for s in seats]
    contrib = [s["bet"] for s in seats]
    folded = [s["folded"] for s in seats]

    hole = [[] for _ in range(len(seats))]
    hero_cards = cards_str_to_ids(hole_cards)
    if len(hero_cards) == 2:
        hole[hero_index] = hero_cards

    board = cards_str_to_ids(community_cards)
    current_bet = max(contrib) if contrib else 0.0
    initial_stacks = tracker.initial_stacks or [stacks[i] + contrib[i] for i in range(len(stacks))]

    state = GameState(
        deck=[],
        board=board,
        hole=hole,
        pot=pot,
        to_act=to_act,
        street=street,
        stacks=stacks,
        current_bet=current_bet,
        last_aggressor=tracker.last_aggressor,
        sb_player=snapshot.get("sb", 0),
        bb_player=snapshot.get("bb", 1),
        button_player=snapshot.get("button", 0),
        initial_stacks=initial_stacks,
        contrib=contrib,
        folded=folded,
        players_acted=tracker.players_acted or [True] * len(stacks),
        num_players=len(stacks),
        actions_this_street=0,
        terminal=False,
        winner=-1,
        action_seq=tracker.action_seq,
    )
    return state, hero_index


def load_policy_net(state_dim, path):
    net = PolicyNet(state_dim)
    state_dict = torch.load(path, map_location="cpu")
    net.load_state_dict(state_dict)
    net.eval()
    return net


def ensure_policy_loaded(num_players, policy_net, policy_env):
    if policy_net is not None and policy_env is not None and policy_env.num_players == num_players:
        return policy_net, policy_env
    policy_env = SimpleHoldemEnv(
        stack_size=STACK_SIZE,
        sb=TABLE_SMALL_BLIND,
        bb=TABLE_BIG_BLIND,
        num_players=num_players,
    )
    dummy = policy_env.new_hand()
    state_dim = encode_state(dummy, 0).shape[0]
    policy_net = load_policy_net(state_dim, POLICY_PATH)
    return policy_net, policy_env


def get_policy_action_probs(policy_net, state, hero_index, legal_actions):
    x = encode_state(state, hero_index).float().unsqueeze(0)
    with torch.no_grad():
        logp = policy_net(x).squeeze(0)
    mask = torch.full((NUM_ACTIONS,), -1e9)
    for a in legal_actions:
        mask[a] = 0.0
    probs = torch.softmax(logp + mask, dim=-1).tolist()
    return probs


def best_action_from_probs(probs, legal_actions):
    if not legal_actions:
        return None
    return max(legal_actions, key=lambda a: probs[a])


def setup_logger():
    logger.handlers = []
    logger.setLevel(logging.INFO)
    logger.propagate = False
    formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    if LOG_TABLE_TO_FILE:
        fh = logging.FileHandler(LOG_FILE_PATH, encoding="utf-8")
        fh.setFormatter(formatter)
        logger.addHandler(fh)
    if LOG_PRINT_CONSOLE:
        sh = logging.StreamHandler()
        sh.setFormatter(formatter)
        logger.addHandler(sh)


def format_policy_probs(probs, legal_actions):
    legal = set(legal_actions)
    parts = []
    for a in range(NUM_ACTIONS):
        name = ACTION_ID_TO_NAME.get(a, f"ACT_{a}")
        if a in legal:
            parts.append(f"{name}={probs[a]:.3f}")
        else:
            parts.append(f"{name}={probs[a]:.3f} (illegal)")
    return "Policy probs: " + ", ".join(parts)


def print_policy_probs(probs, legal_actions):
    msg = format_policy_probs(probs, legal_actions)
    if LOG_PRINT_CONSOLE:
        print(msg)
    if LOG_FULL_STATE:
        logger.info(msg)


def log_event(message, payload=None):
    if not LOG_FULL_STATE:
        return
    if payload is None:
        logger.info(message)
    else:
        logger.info(json.dumps({"message": message, "payload": payload}, ensure_ascii=True))


def state_vector_summary(state, hero_index):
    try:
        vec = encode_state(state, hero_index).float()
        return {
            "len": int(vec.shape[0]),
            "min": float(vec.min().item()),
            "max": float(vec.max().item()),
            "mean": float(vec.mean().item()),
        }
    except Exception as e:
        return {"error": str(e)}


def log_full_snapshot(snapshot, hole_cards, community_cards, pot, equity, cct, state, legal_actions, probs):
    if not LOG_FULL_STATE:
        return
    hero_index = snapshot.get("hero_index")
    to_act = snapshot.get("to_act")
    seats = snapshot.get("seats", [])
    seat_dump = []
    for s in seats:
        seat_dump.append({
            "seat_index": s.get("seat_index"),
            "name": s.get("name"),
            "stack": s.get("stack"),
            "bet": s.get("bet"),
            "folded": s.get("folded"),
            "is_hero": s.get("is_hero"),
            "is_button": s.get("is_button"),
            "is_sb": s.get("is_sb"),
            "is_bb": s.get("is_bb"),
            "is_active": s.get("is_active"),
            "raw_stack_text": s.get("raw_stack_text"),
            "raw_bet_text": s.get("raw_bet_text"),
            "raw_name_text": s.get("raw_name_text"),
            "class_name": s.get("class_name"),
            "action_text": s.get("action_text"),
        })
    to_call = None
    hero_stack = None
    if state is not None and hero_index is not None and hero_index >= 0:
        to_call = max(0.0, state.current_bet - state.contrib[hero_index])
        hero_stack = state.stacks[hero_index]
    probs_named = {}
    if probs is not None:
        for a in range(NUM_ACTIONS):
            probs_named[ACTION_ID_TO_NAME.get(a, f"ACT_{a}")] = probs[a]
    sum_bets = sum(s.get("bet") or 0.0 for s in seats)
    sum_stacks = sum(s.get("stack") or 0.0 for s in seats)
    payload = {
        "hero_index": hero_index,
        "to_act": to_act,
        "num_players": snapshot.get("num_players"),
        "button": snapshot.get("button"),
        "sb": snapshot.get("sb"),
        "bb": snapshot.get("bb"),
        "seat_elem_count": snapshot.get("seat_elem_count"),
        "pot": pot,
        "sum_bets": sum_bets,
        "sum_stacks": sum_stacks,
        "equity": equity,
        "cost_to_call": cct,
        "hole_cards": hole_cards,
        "community_cards": community_cards,
        "legal_actions": legal_actions,
        "probs": probs_named,
        "to_call": to_call,
        "hero_stack": hero_stack,
        "state": None,
        "state_vec": None,
        "seats": seat_dump,
    }
    if state is not None:
        payload["state"] = {
            "street": state.street,
            "pot": state.pot,
            "current_bet": state.current_bet,
            "last_aggressor": state.last_aggressor,
            "stacks": state.stacks,
            "contrib": state.contrib,
            "folded": state.folded,
            "players_acted": state.players_acted,
            "action_seq": state.action_seq,
            "board": state.board,
            "hero_hole": state.hole[hero_index] if hero_index is not None else None,
        }
        payload["state_vec"] = state_vector_summary(state, hero_index)
    logger.info(json.dumps(payload, ensure_ascii=True))


def calculate_safe_bet(pot_size, equity):
    """
    Calculate the safe bet amount to be profitable in the long run.
    
    :param pot_size: float - Current pot size.
    :param equity: float - Your equity (0.0 to 1.0).
    :return: float - Safe bet amount.
    """
    if equity <= 0 or equity >= 1:
        # If equity is 0 or 1, bet amount doesn't make sense (all-in or fold scenarios).
        print("Invalid equity value. Must be between 0 and 1.")
        return 0.0

    try:
        # Calculate safe bet amount using the formula
        safe_bet = pot_size * (equity / (1 - equity))
        print(f"Calculated Safe Bet: {safe_bet} for Pot Size: {pot_size}, Equity: {equity}")
        return safe_bet
    except ZeroDivisionError:
        print("Equity cannot be exactly 1.")
        return pot_size  # Go all-in if equity is near 1

def calculate_ev(pot_size, win_probability, cost_to_call):
    """
    Calculate the Expected Value (EV) for a given poker decision.
    :param pot_size: float - The current total pot size.
    :param win_probability: float - The probability of winning (0.0 to 1.0).
    :param cost_to_call: float - The cost to call the current bet.
    :return: float - The calculated Expected Value (EV).
    """
    ev = (pot_size * win_probability) - cost_to_call
    print(f"EV Calculation: Pot Size = {pot_size}, Win Probability = {win_probability}, Cost to Call = {cost_to_call}, EV = {ev}")
    return pot_size

# def read_pot_size(driver):
#     """
#     Extract the total pot size from the table.
#     """
#     try:
#         # Locate the span with class "total-pot-amount"
#         pot_size_element = driver.find_element(By.CLASS_NAME, "total-pot-amount")
        
#         # Extract the text and clean it up
#         pot_size_text = pot_size_element.text.strip()
        
#         # Remove the currency symbol (e.g., "€") and convert to a float
#         pot_size = float(pot_size_text.replace("€", "").strip())
        
#         print(f"Total Pot Size: {pot_size}")
#         return pot_size
#     except Exception as e:
#         print(f"Error extracting pot size: {e}")
#         return None
# def read_pot_size(driver):
#     """
#     Extract the total pot size and previous pot size from the table.
#     """
#     try:
#         # # Locate the span with class "total-pot-amount" for the current total pot size
#         # total_pot_element = driver.find_element(By.CLASS_NAME, "total-pot-amount")
#         # total_pot_text = total_pot_element.text.strip()
#         # total_pot = float(total_pot_text.replace("€", "").strip())

#         # print(f"Total Pot Size: {total_pot}")

#         # Locate the div with id "main-pot" for the previous pot size
#         previous_pot_element = driver.find_element(By.ID, "main-pot")
#         previous_pot_text = previous_pot_element.text.strip()
#         previous_pot = float(previous_pot_text.replace("€", "").strip())

#         print(f"Previous Pot Size: {previous_pot}")
#         return previous_pot
#     except Exception as e:
#         print(f"Error extracting pot size: {e}")
#         return None


def read_pot_size(driver):
    """
    Extract the current pot size from the table.
    """
    try:
        total_pot_element = driver.find_element(By.CSS_SELECTOR, ".total-pot-amount")
        total_text = total_pot_element.text.strip()
        if total_text:
            total_pot = parse_money(total_text)
            print(f"Total Pot Size: {total_pot:.2f}")
            return total_pot
    except Exception:
        pass

    try:
        amount_cont_element = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "amount-cont"))
        )
        print("Found 'amount-cont' element.")
        print("Inner HTML of 'amount-cont':", amount_cont_element.get_attribute("innerHTML"))

        try:
            main_pot_element = amount_cont_element.find_element(By.ID, "main-pot")
            print("Found 'main-pot' element.")
        except NoSuchElementException:
            print("'main-pot' not found within 'amount-cont'. Attempting global search.")
            main_pot_element = driver.find_element(By.ID, "main-pot")

        main_pot_text = main_pot_element.text.strip()
        print(f"Raw pot text: {main_pot_text}")
        main_pot = parse_money(main_pot_text)
        print(f"Previous Pot Size: {main_pot:.2f}")
        return main_pot

    except Exception as e:
        print(f"Error extracting previous pot size: {e}")
        print(f"Page source at the time of error: {driver.page_source}")
        return 0.0

def get_cost_to_call(driver):
    """
    Extract the cost to call from the DOM.
    :param driver: Selenium WebDriver instance.
    :return: float - Cost to Call.
    """
    try:
        call_button = driver.find_element(By.ID, "CALL")
        cost_value_element = call_button.find_element(By.CLASS_NAME, "action-value")
        cost_to_call_text = cost_value_element.text.strip()
        cost_to_call = parse_money(cost_to_call_text)
        return cost_to_call
    except Exception as e:
        print(f"Error extracting Cost to Call: {e}")
        return 0.0

def card_to_number(card):
    """
    Convert card face values to numbers.
    """
    card_values = {"J": 11, "Q": 12, "K": 13, "A": 14}
    if card in card_values:
        return card_values[card]  # Return the mapped value for face cards
    try:
        return int(card)  # Convert numeric card values (e.g., '10', '8') to integers
    except ValueError:
        print(f"Invalid card value: '{card}'")  # Log invalid card value
        return None

def read_cards(driver):
    """
    Read the card values and return as numbers.
    """
    try:
        # Locate the card elements
        card1 = driver.find_element(By.CSS_SELECTOR, ".card1 .card-rank").text.strip()
        card2 = driver.find_element(By.CSS_SELECTOR, ".card2 .card-rank").text.strip()
        print(f"Raw card values: {card1}, {card2}")  # Debug log for raw card values

        
        # Convert card values to numbers
        card1_number = card_to_number(card1)
        card2_number = card_to_number(card2)
        
        return card1_number, card2_number
    except Exception as e:
        print(f"Error reading cards: {e}")
        return None, None

def gen_cards(cards_str):
    """
    Generate Card objects from a list of card strings.
    Card strings must be in the format 'CA' (e.g., 'CA' = Ace of Clubs).
    """
    suit_map = {
        "C": Card.CLUB,
        "H": Card.HEART,
        "S": Card.SPADE,
        "D": Card.DIAMOND
    }
    rank_map = {
        "2": 2, "3": 3, "4": 4, "5": 5, "6": 6, "7": 7, "8": 8, "9": 9, 
        "10": 10, "T": 10, "J": 11, "Q": 12, "K": 13, "A": 14
    }

    try:
        cards = []
        for card_str in cards_str:
            if len(card_str) == 3:  # Handle "10" case, e.g., "10S"
                rank = "10"
                suit = card_str[2]
            else:
                rank = card_str[0]  # Extract the rank (first character)
                suit = card_str[1]  # Extract the suit (second character)
            
            suit = suit_map[suit.upper()]  # Translate suit to Card enum
            rank = rank_map[rank.upper()]  # Translate rank to integer
            
            cards.append(Card(suit, rank))

        return cards
    except Exception as e:
        print(f"Error in gen_cards: {e}")
        return []

def read_cards_2(driver):
    """
    Read the card values and return them as a list of parsed card strings.
    """
    try:
        # Locate the card elements
        hole_cards = []
        card_elements = driver.find_elements(By.CSS_SELECTOR, ".cards-holder-hero .card-wrapper")  # Update with correct CSS selector
        
        for card_elem in card_elements:
            # Extract rank
            rank = card_elem.find_element(By.CLASS_NAME, "card-rank").text.strip()
            
            # Extract suit
            suit = card_elem.find_element(By.CLASS_NAME, "card-suit").text.strip()
            translated_suit = translate_suit(suit)  # Translate suit symbol to a letter (C, H, S, D)

            if rank and translated_suit:
                card_string = f"{rank.upper()}{translated_suit}"  # Combine rank and suit (e.g., "AC")
                hole_cards.append(card_string)
        
        print(f"Hole cards: {hole_cards}")
        return hole_cards
    except Exception as e:
        print(f"Error reading hole cards: {e}")
        return []

def read_community_cards(driver):
    """
    Read the community cards from the table.
    Returns a list of card strings (e.g., ["5D", "JS", "7H"]) if present.
    If no community cards are present, returns an empty list without raising an error.
    """
    try:
        # Locate the community card elements
        community_cards = []
        card_elements = driver.find_elements(By.CSS_SELECTOR, ".community-cards .card-wrapper")  # Adjust the CSS selector as needed
        
        for card_elem in card_elements:
            # Extract rank
            rank = card_elem.find_element(By.CLASS_NAME, "card-rank").text.strip()
            
            # Extract suit
            suit = card_elem.find_element(By.CLASS_NAME, "card-suit").text.strip()
            translated_suit = translate_suit(suit)  # Translate suit symbol to a letter (C, H, S, D)

            if rank and translated_suit:
                card_string = f"{rank.upper()}{translated_suit}"  # Combine rank and suit (e.g., "AC")
                community_cards.append(card_string)

        if not community_cards:
            print("No community cards are present yet.")
            return []
        
        print(f"Community cards: {community_cards}")
        return community_cards
    except Exception as e:
        print(f"Error reading community cards")
        return []

def translate_suit(suit_symbol):
    """
    Translate card suit symbol to its corresponding letter (C, H, S, D).
    """
    suit_map = {
        "♣": "C",  # Clubs
        "♦": "D",  # Diamonds
        "♥": "H",  # Hearts
        "♠": "S"   # Spades
    }
    return suit_map.get(suit_symbol, None)


def is_folded(driver):
    """
    Check if the player has folded.
    """
    try:
        # Locate the folded cards container
        folded_container = driver.find_element(By.CSS_SELECTOR, ".hero-folded-cards")
        return folded_container.is_displayed()
    except Exception as e:
        # If the folded element is not found, assume not folded
        return False


# def make_decision(driver, card1_number, card2_number, action_button):
#     """
#     Make a decision based on the cards: beep or fold.
#     """
#     try:
#         if card1_number > 9 and card2_number > 9:
#             print(f"Cards: {card1_number} and {card2_number}. Beep!")
#             winsound.Beep(1400, 500)  # 1000 Hz for 500 ms
#             winsound.Beep(1400, 500)  # 1000 Hz for 500 ms
#             time.sleep(10)  # Adjust this based on how long it takes to start the game

#         else:
#             print(f"Cards: {card1_number} and {card2_number}. Folding...")
#             if action_button.get_attribute("id") == "FOLD":
#                 action_button.click()
#     except Exception as e:
#         print(f"Error making decision: {e}")
def make_decision(driver, card1_number, card2_number, action_button,equity, cct, ev, safe_bet):
    """
    Make a decision based on the cards: beep or fold. Use the pre-action wrapper if necessary.
    """
    try:
        if equity> 0.20 or card1_number > 9 and card2_number > 9 or card1_number == card2_number or (card1_number > 12 and card2_number >7) or (card2_number > 12 and card1_number >7):
            print(f"Cards: {card1_number} and {card2_number}. Beep!")
            winsound.Beep(1400, 500)  # 1400 Hz for 500 ms
            time.sleep(3)  # Adjust based on game timing
        # if equity > 0.37:
        #     winsound.Beep(1400, 500)  # 1400 Hz for 500 ms
        #     winsound.Beep(1400, 500)  # Another beep
        #     time.sleep(10)  # Adjust based on game timing
            
        else:
            print(f"Cards: {card1_number} and {card2_number}. Folding...")
            try:
                # Attempt to click the FOLD button
                if action_button.get_attribute("id") == "FOLD":
                    action_button.click()
            except Exception as e:
                print(f"Error clicking FOLD button. Attempting to use pre-action wrapper...")
                # Fall back to the pre-action wrapper for Check/Fold
                try:
                    pre_action_checkbox = driver.find_element(By.ID, "FOLD")  # Adjust ID if needed
                    pre_action_checkbox.click()
                    print("Pre-action Check/Fold activated.")
                except Exception as e2:
                    print(f"Error activating pre-action Check/Fold")
    except Exception as e:
        print(f"Error making decision: {e}")


def wait_for_actionable_buttons(driver):
    """
    Wait for one of the actionable buttons (FOLD, CHECK, RAISE_TO) to appear.
    """
    try:
        wait = WebDriverWait(driver, 10)  # Wait up to 30 seconds
        # Check for FOLD, CHECK, or RAISE_TO buttons individually
        for button_id in ["FOLD", "CHECK", "RAISE_TO"]:
            try:
                action_button = wait.until(EC.presence_of_element_located((By.ID, button_id)))
                return action_button  # Return the first found actionable button
            except Exception:
                pass  # If this button is not found, continue checking others
        print("No actionable buttons detected.")
        return None
    except Exception as e:
        print(f"Error waiting for actionable buttons: {e}")
        return None


def main():
    """
    Main function to run the poker automation script.
    """
    global LOG_TICK
    # Launch the browser and navigate to the poker game
    driver = webdriver.Chrome()  # Ensure you have ChromeDriver installed and in PATH
    driver.maximize_window()
    # https://mgames-poker-fr3.williamhill.com/poker/web/25.1.1.57_1/html/poker/index.html?launcherRedirect=true&hostedMode=3
    driver.get("https://mgames-poker-fr3.williamhill.com/poker/web/25.1.1.57_1/html/poker/index.html?launcherRedirect=true&hostedMode=3")
    setup_logger()
    if LOG_TABLE_TO_FILE:
        print(f"Logging to {LOG_FILE_PATH}")
    log_event("Logging initialized", {"log_file": LOG_FILE_PATH, "log_console": LOG_PRINT_CONSOLE})
    policy_net = None
    policy_env = None
    table_tracker = TableTracker()

    # Wait for the user to log in and start the game
    print("Log in and start a game. Waiting for 20 seconds...")
    time.sleep(10)  # Adjust this based on how long it takes to start the game
    print('Now waiting for Actionable Buttons')
    try:
        # Run the script indefinitely until the browser is closed
        while True:
            LOG_TICK += 1
            # Wait for actionable buttons to appear
            action_button = wait_for_actionable_buttons(driver)
          
            if not action_button:
                print("No actionable buttons detected. Retrying...")
                log_event("No actionable buttons detected")
                time.sleep(random.uniform(1, 2))  # Sleep for a random time between 1 and 4 seconds
                continue
            
            # Check if the player is folded
            if is_folded(driver):
                print("Player is fold. Waiting for next round...")
                log_event("Player is folded; waiting for next round")
                time.sleep(random.uniform(1, 2))  # Sleep for a random time between 1 and 4 seconds

            hole_cards = read_cards_2(driver)  # Returns something like ["AC", "KH"]
            community_cards = read_community_cards(driver)  # Returns something like ["5D", "JS", "7H"]
            
            if len(hole_cards) == 2:
                # Parse cards for equity calculation
                hole_card_objects = gen_cards(hole_cards)
                community_card_objects = gen_cards(community_cards) if community_cards else []

                # Calculate equity
                if len(community_cards) == 0:  # Pre-flop
                    equity = estimate_hole_card_win_rate(
                        nb_simulation=1500,
                        nb_player=6,
                        hole_card=hole_card_objects,
                        community_card=[]
                    )
                else:  # Post-flop, turn, or river
                    equity = estimate_hole_card_win_rate(
                        nb_simulation=1500,
                        nb_player=6,
                        hole_card=hole_card_objects,
                        community_card=community_card_objects
                    )
                cct = get_cost_to_call(driver)
                potsize = read_pot_size(driver)
                ev = calculate_ev(potsize, equity, cct)
                safe_bet = calculate_safe_bet(potsize, equity)

                if LOG_PRINT_CONSOLE:
                    print(potsize)
                log_event("Metrics", {"pot": potsize, "equity": equity, "cost_to_call": cct, "ev": ev, "safe_bet": safe_bet})
                update_interactive_display(equity, hole_cards, community_cards)

                snapshot = read_table_snapshot(driver, force_hero_to_act=True)
                if snapshot:
                    policy_net, policy_env = ensure_policy_loaded(snapshot["num_players"], policy_net, policy_env)
                    state, hero_index = build_game_state(snapshot, hole_cards, community_cards, potsize, table_tracker)
                    if state is not None:
                        legal_actions = policy_env.legal_actions(state)
                        probs = get_policy_action_probs(policy_net, state, hero_index, legal_actions)
                        print_policy_probs(probs, legal_actions)
                        if LOG_TICK % LOG_EVERY_N == 0:
                            log_full_snapshot(snapshot, hole_cards, community_cards, potsize, equity, cct, state, legal_actions, probs)
                    else:
                        log_event("State build failed", {"snapshot": snapshot})
                else:
                    log_event("No table snapshot; adjust selectors.")

            else:
                print("Unable to read hole cards. Retrying...")
                log_event("Unable to read hole cards", {"hole_cards": hole_cards})
                time.sleep(random.uniform(1, 2))
                continue

            if USE_POLICY_ONLY:
                time.sleep(random.uniform(1, 2))
                continue

            card1_number, card2_number = read_cards(driver)
            if card1_number and card2_number:
                print(f"Dealt cards: {card1_number} and {card2_number}")

                # Make a decision based on the cards and action button
                make_decision(driver, card1_number, card2_number, action_button, equity, cct, ev, safe_bet)
                time.sleep(random.uniform(1, 2))
                
            else:
                print("Unable to read cards. Retrying...")


    except KeyboardInterrupt:
        print("Script terminated by user.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
    finally:
        # Close the browser when the script ends
        # driver.quit()
        print('exit')


if __name__ == "__main__":
    main()


In [ ]:

equity = estimate_hole_card_win_rate(
                        nb_simulation=1500,
                        nb_player=6,
                        hole_card=hole_card_objects,
                        community_card=community_card_objects
                    )